# Parameters vs Hyperparameters

**Concept**
Parameters are values the model *learns* from the training data (e.g. weights, coefficients, intercepts). Hyperparameters are values *we set before training* that control how the learning happens (e.g. `C`, `max_depth`, `n_neighbors`).

**Why it is required / what problem it solves**
Confusing the two leads to confusion about what actually changes during training. Parameters change automatically as the model fits the data; hyperparameters stay fixed during training and must be chosen (or tuned) by the engineer.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

# Hyperparameters: chosen by us BEFORE training
model = LogisticRegression(C=1.0, max_iter=5000)
model.fit(X_train, y_train)

# Parameters: learned by the model DURING training
print("Hyperparameter (set by us) -> C:", model.C)
print("Learned parameter (first 3 weights) ->", model.coef_[0][:3])
print("Learned parameter (bias/intercept) ->", model.intercept_)


Hyperparameter (set by us) -> C: 1.0
Learned parameter (first 3 weights) -> [ 0.98617021  0.22632086 -0.36882765]
Learned parameter (bias/intercept) -> [29.47958439]


**What this example tells us**
The hyperparameter `C` stays exactly as we set it (`1.0`) before and after training, while the coefficients and intercept are learned values that only exist *after* calling `.fit()`.

**AI/ML Example**
In an image classifier, the learning rate is a hyperparameter you pick beforehand; the millions of network weights are parameters learned during training.

**Business Example**
In a loan-approval model, an engineer sets `C` (a hyperparameter) before training; the resulting coefficients (parameters) tell you how much each factor like income or credit score actually influenced the decision.

**AI/ML Engineer Use Case**
When debugging a model, always check whether an issue comes from a misconfigured hyperparameter (your choice) or a parameter (the model's output) — they need very different fixes.

# Why Hyperparameters Matter

**Concept**
Hyperparameters directly control model complexity — how flexible or restrictive the model is allowed to be while learning.

**Why it is required / what problem it solves**
The right hyperparameters balance underfitting (too simple, poor accuracy everywhere) and overfitting (too complex, memorizes training data but fails on new data). Picking them well is often more impactful than the choice of algorithm itself.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

for depth in [1, 5, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"max_depth={str(depth):>4} -> train acc={train_acc:.3f}, test acc={test_acc:.3f}")


max_depth=   1 -> train acc=0.921, test acc=0.895
max_depth=   5 -> train acc=0.996, test acc=0.947
max_depth=None -> train acc=1.000, test acc=0.947


**What this example tells us**
As `max_depth` increases, training accuracy keeps rising toward 100%, but test accuracy plateaus — the deeper tree starts overfitting rather than genuinely improving.

**AI/ML Example**
In fraud detection, an overly deep tree can "memorize" specific fraudulent transactions from the training set but fail to catch new, slightly different fraud patterns.

**Business Example**
A recommendation engine tuned to fit historical purchases too tightly may recommend well on past data but perform poorly for new customers.

**AI/ML Engineer Use Case**
Watch the gap between train and test accuracy while adjusting hyperparameters — a growing gap is your signal to add more regularization or reduce model complexity.

# Model Configuration

**Concept**
"Configuring" a model means setting its hyperparameters, either when creating the model object or afterward using `set_params()`. Every model ships with sensible defaults, viewable via `get_params()`.

**Why it is required / what problem it solves**
Knowing how to inspect and change a model's configuration lets you experiment systematically instead of guessing, and makes it easy to plug models into tuning tools like `GridSearchCV`.

In [1]:
from sklearn.svm import SVC

model = SVC()

# See all default hyperparameters (the "configuration" of the model)
print("Default configuration:")
print(model.get_params())

# Change configuration before training
model.set_params(C=10, kernel="rbf", gamma="scale")
print("\nUpdated C:", model.C)
print("Updated kernel:", model.kernel)


Default configuration:
{'C': 1.0, 'break_ties': False, 'cache_size': 200, 'class_weight': None, 'coef0': 0.0, 'decision_function_shape': 'ovr', 'degree': 3, 'gamma': 'scale', 'kernel': 'rbf', 'max_iter': -1, 'probability': False, 'random_state': None, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Updated C: 10
Updated kernel: rbf


**What this example tells us**
`get_params()` shows every hyperparameter a model supports along with its default value, and `set_params()` lets you change them without recreating the object.

**AI/ML Example**
Automated hyperparameter search tools (like `GridSearchCV`) work by repeatedly calling `set_params()` internally to try many configurations.

**Business Example**
A data science team maintaining a production model can log the exact `get_params()` output alongside each model version for full reproducibility.

**AI/ML Engineer Use Case**
Always print `get_params()` before tuning — it prevents wasting time tuning a hyperparameter that doesn't exist or is already at a sensible default.

# Common Hyperparameters — Decision Tree

**Concept**
- `max_depth`: maximum depth the tree is allowed to grow to.
- `min_samples_split`: minimum number of samples a node must have before it can be split further.
- `min_samples_leaf`: minimum number of samples required to keep a leaf node.
- `criterion`: function used to measure the quality of a split (e.g. `"gini"`, `"entropy"`).

**Why it is required / what problem it solves**
Without limits like `max_depth` or `min_samples_leaf`, a decision tree keeps splitting until it perfectly memorizes the training data — a classic overfitting trap. These hyperparameters constrain the tree's growth.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(
    max_depth=4,          # limits how deep the tree can grow
    min_samples_split=10, # min samples needed to split a node
    min_samples_leaf=5,   # min samples required in a leaf node
    criterion="gini"      # function used to measure split quality
)
model.fit(X_train, y_train)
print("Test accuracy:", accuracy_score(y_test, model.predict(X_test)))
print("Actual tree depth used:", model.get_depth())


Test accuracy: 0.9473684210526315
Actual tree depth used: 4


**What this example tells us**
Limiting the tree with `max_depth`, `min_samples_split`, and `min_samples_leaf` keeps the tree small and controlled — the printed depth confirms the tree respected the limit we set.

**AI/ML Example**
In medical diagnosis models, a shallow, constrained tree is often preferred since it stays interpretable to doctors while still being accurate.

**Business Example**
A churn-prediction tree with `min_samples_leaf` set high avoids creating tiny, unreliable leaf nodes based on just 1-2 customers.

**AI/ML Engineer Use Case**
Start with a shallow tree and increase `max_depth` gradually while watching test accuracy — this is a fast way to find a good complexity level before trying more expensive models.

# Common Hyperparameters — Random Forest

**Concept**
- `n_estimators`: number of individual trees in the forest.
- `max_depth`: maximum depth of each individual tree.
- `max_features`: number of features considered when looking for the best split.
- `min_samples_split`: minimum samples required to split a node in each tree.

**Why it is required / what problem it solves**
A random forest is many trees combined. `n_estimators` controls how many "votes" you get (more trees = more stable predictions, at the cost of speed), while the other hyperparameters control how diverse and how complex each individual tree is.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

for n_trees in [1, 10, 100]:
    model = RandomForestClassifier(
        n_estimators=n_trees,  # number of trees in the forest
        max_depth=5,           # max depth of each tree
        max_features="sqrt",   # number of features considered per split
        min_samples_split=4,   # min samples needed to split a node
        random_state=42
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"n_estimators={n_trees:>3} -> test accuracy={acc:.3f}")


n_estimators=  1 -> test accuracy=0.939
n_estimators= 10 -> test accuracy=0.956
n_estimators=100 -> test accuracy=0.965


**What this example tells us**
Accuracy improves as `n_estimators` increases from 1 to 100, but the gains shrink — more trees help up to a point, after which they mostly add computation cost.

**AI/ML Example**
In click-through-rate prediction at scale, teams balance `n_estimators` against latency requirements, since more trees mean slower predictions.

**Business Example**
A credit risk model using a random forest with too few trees may give unstable risk scores that change a lot between retrains; more trees stabilize the output.

**AI/ML Engineer Use Case**
Use a moderate `n_estimators` (e.g. 100-300) as a strong baseline, then tune `max_depth` and `max_features` rather than blindly adding more trees for marginal gains.

# Common Hyperparameters — KNN

**Concept**
- `n_neighbors`: how many nearest neighbors are used to make a prediction.
- `weights`: whether all neighbors count equally (`"uniform"`) or closer ones count more (`"distance"`).
- `metric`: the distance function used to decide who counts as a "neighbor" (e.g. `"minkowski"`, `"euclidean"`).

**Why it is required / what problem it solves**
KNN has no real training phase — its behavior is entirely defined by these hyperparameters at prediction time. Too few neighbors -> noisy predictions; too many -> overly smoothed, less accurate boundaries.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

for k in [1, 5, 15]:
    model = KNeighborsClassifier(
        n_neighbors=k,     # how many nearest neighbors to vote
        weights="distance",# closer neighbors count more than far ones
        metric="minkowski" # distance function used to find neighbors
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"n_neighbors={k:>2} -> test accuracy={acc:.3f}")


n_neighbors= 1 -> test accuracy=0.939
n_neighbors= 5 -> test accuracy=0.947
n_neighbors=15 -> test accuracy=0.956


**What this example tells us**
Accuracy changes with `n_neighbors` — very small `k` (like 1) can be sensitive to noisy points, while a well-chosen `k` (like 15 here) generalizes better on this dataset.

**AI/ML Example**
In a product-recommendation system based on similarity, `n_neighbors` controls how many similar users influence a recommendation.

**Business Example**
A customer segmentation tool using KNN with too small a `k` might classify a customer based on one unusual neighbor rather than a representative group.

**AI/ML Engineer Use Case**
Always scale features (as done here with `StandardScaler`) before using KNN — distance-based hyperparameters like `n_neighbors` only work correctly when features are on comparable scales.

# Common Hyperparameters — SVM

**Concept**
- `C`: penalty for misclassified points; controls how strict the margin is (small `C` = wider margin, more tolerant of errors; large `C` = stricter, fits training data tighter).
- `kernel`: the function used to separate classes (`"linear"`, `"rbf"`, `"poly"`, etc.).
- `gamma`: how far a single training point's influence reaches (only used for non-linear kernels like `"rbf"`).

**Why it is required / what problem it solves**
These hyperparameters shape the decision boundary. Getting `C` and `gamma` wrong is one of the most common reasons an SVM underperforms — a low `C` can cause severe underfitting, as shown in the example below.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

for c_val in [0.01, 1, 100]:
    model = SVC(
        C=c_val,          # penalty for misclassifying points (margin strictness)
        kernel="rbf",      # function used to separate classes
        gamma="scale"      # how far the influence of one point reaches
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"C={c_val:>6} -> test accuracy={acc:.3f}")


C=  0.01 -> test accuracy=0.623
C=     1 -> test accuracy=0.982
C=   100 -> test accuracy=0.939


**What this example tells us**
A very small `C` (0.01) causes severe underfitting (63% accuracy), while `C=1` performs best and `C=100` starts to overfit slightly — `C` is one of the most sensitive hyperparameters to tune.

**AI/ML Example**
In text classification with an SVM, a poorly chosen `C` can make the model ignore almost all signal in the data, as seen in the low-accuracy result above.

**Business Example**
A spam filter built with SVM needs `C` tuned carefully — too permissive (low `C`) misses obvious spam, too strict (high `C`) may overfit to specific past spam wording.

**AI/ML Engineer Use Case**
Always tune `C` (and `gamma` for non-linear kernels) using cross-validation rather than default values — as shown, defaults can be far from optimal for a given dataset.

# Common Hyperparameters — Logistic Regression

**Concept**
- `C`: inverse of regularization strength (smaller `C` = stronger regularization = simpler model).
- `penalty`: type of regularization applied to the weights (e.g. `"l2"`, `"l1"`).
- `solver`: optimization algorithm used to find the best weights (e.g. `"lbfgs"`, `"liblinear"`).

**Why it is required / what problem it solves**
Regularization (`C`, `penalty`) prevents the model's weights from growing too large and overfitting, while `solver` affects how efficiently and reliably training converges, especially on larger datasets.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

for c_val in [0.01, 1, 10]:
    model = LogisticRegression(
        C=c_val,           # inverse of regularization strength
        penalty="l2",       # type of regularization applied to weights
        solver="lbfgs",     # optimization algorithm used to fit the model
        max_iter=5000
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"C={c_val:>5} -> test accuracy={acc:.3f}")


C= 0.01 -> test accuracy=0.965
C=    1 -> test accuracy=0.974
C=   10 -> test accuracy=0.974


**What this example tells us**
Increasing `C` from 0.01 to 1 improves accuracy, but returns diminish past that point — moderate regularization strength works best for this dataset.

**AI/ML Example**
In email classification, `penalty="l2"` keeps all word-features contributing a little, while `penalty="l1"` can zero out irrelevant word-features entirely for a sparser model.

**Business Example**
A marketing team predicting customer conversion uses `C` to control how much the model trusts noisy signals from a small ad campaign dataset.

**AI/ML Engineer Use Case**
When a logistic regression model fails to converge, check `solver` first — some solvers only support certain `penalty` types, and mismatches cause training errors.